# 3D Simulation: Sphere in a Fluid Flow

Inviscid potential flow past a sphere (uniform free stream + dipole).

Velocity (in spherical coordinates, axis along $x$):

$$
v_r = U\cos\theta\left(1 - \frac{R^3}{r^3}\right),\qquad
v_\theta = -U\sin\theta\left(1 + \frac{R^3}{2r^3}\right)
$$

Surface pressure coefficient from Bernoulli: $C_p = 1 - \left(\tfrac{3}{2}\sin\theta\right)^2$.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

np.set_printoptions(precision=4, suppress=True)
print("numpy", np.__version__)

## 1. Flow & domain parameters

In [ ]:
# Free-stream speed along +x, sphere radius, domain size
U_INF = 1.0
R = 1.0
DOMAIN = 4.0          # half-box extent in each direction
RHO = 1.0             # density (for pressure)
P_INF = 0.0           # freestream pressure

N_GRID = 36           # Cartesian sample grid for |u| slices
N_STREAM = 48         # number of 3D streamlines
N_PARTICLES = 120     # tracer particles for animation
DT = 0.04
N_FRAMES = 80

print(f"Re conceptual (inviscid): U={U_INF}, R={R}, domain=±{DOMAIN}")

## 2. Analytical velocity field (potential flow)

In [ ]:
def velocity_at(x, y, z, U=U_INF, R=R, eps=1e-12):
    """Potential-flow velocity around a sphere centered at the origin.

    Returns (u, v, w) with the same shape as the inputs.
    Inside / on the sphere the velocity is set to zero (solid body).
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    z = np.asarray(z, dtype=float)

    r2 = x * x + y * y + z * z
    r = np.sqrt(r2 + eps)
    inside = r <= R

    # Spherical basis: flow axis = x
    cos_th = x / r
    sin_th = np.sqrt(np.clip(1.0 - cos_th * cos_th, 0.0, 1.0))

    # Radial and polar components
    R3_r3 = (R / r) ** 3
    vr = U * cos_th * (1.0 - R3_r3)
    vth = -U * sin_th * (1.0 + 0.5 * R3_r3)

    # Unit vectors e_r, e_theta (theta measured from +x)
    # e_r = r_hat
    er_x, er_y, er_z = x / r, y / r, z / r

    # e_theta lies in the plane of e_r and e_x, perpendicular to e_r
    # direction of increasing theta: toward -x in the meridional plane
    rho_xy = np.sqrt(y * y + z * z) + eps
    eth_x = -sin_th
    eth_y = cos_th * (y / rho_xy)
    eth_z = cos_th * (z / rho_xy)

    # On the x-axis, sin_th ~ 0 → set transverse components safely
    on_axis = rho_xy < 1e-10
    eth_y = np.where(on_axis, 0.0, eth_y)
    eth_z = np.where(on_axis, 0.0, eth_z)

    u = vr * er_x + vth * eth_x
    v = vr * er_y + vth * eth_y
    w = vr * er_z + vth * eth_z

    u = np.where(inside, 0.0, u)
    v = np.where(inside, 0.0, v)
    w = np.where(inside, 0.0, w)
    return u, v, w


def speed(x, y, z, **kw):
    u, v, w = velocity_at(x, y, z, **kw)
    return np.sqrt(u * u + v * v + w * w)


def pressure(x, y, z, U=U_INF, rho=RHO, p_inf=P_INF, **kw):
    """Bernoulli pressure: p = p_inf + 1/2 rho (U^2 - |V|^2)."""
    s = speed(x, y, z, U=U, **kw)
    return p_inf + 0.5 * rho * (U * U - s * s)


# Sanity check at a far-field point and on the equator
u_far, v_far, w_far = velocity_at(10.0, 0.0, 0.0)
u_eq, v_eq, w_eq = velocity_at(0.0, R, 0.0)
print(f"Far field (~U i): ({u_far:.4f}, {v_far:.4f}, {w_far:.4f})")
print(f"Equator surface (|V| should be 1.5 U): speed={speed(0.0, R + 1e-9, 0.0):.4f}")

## 3. Sphere mesh + surface $C_p$

In [ ]:
def sphere_mesh(radius=R, n_theta=48, n_phi=96):
    theta = np.linspace(0.0, np.pi, n_theta)
    phi = np.linspace(0.0, 2.0 * np.pi, n_phi)
    th, ph = np.meshgrid(theta, phi, indexing="ij")
    # Standard math: x = r cosθ (flow axis), y/z transverse
    xs = radius * np.cos(th)
    ys = radius * np.sin(th) * np.cos(ph)
    zs = radius * np.sin(th) * np.sin(ph)
    # Analytical Cp on the surface
    cp = 1.0 - (1.5 * np.sin(th)) ** 2
    return xs, ys, zs, cp


xs, ys, zs, cp_surf = sphere_mesh()
print(f"Sphere mesh: {xs.shape}, Cp range [{cp_surf.min():.3f}, {cp_surf.max():.3f}]")

## 4. Integrate 3D streamlines (RK4)

In [ ]:
def rk4_step(pos, dt, vel_fn=velocity_at):
    x, y, z = pos
    k1 = np.array(vel_fn(x, y, z))
    k2 = np.array(vel_fn(*(pos + 0.5 * dt * k1)))
    k3 = np.array(vel_fn(*(pos + 0.5 * dt * k2)))
    k4 = np.array(vel_fn(*(pos + dt * k3)))
    return pos + (dt / 6.0) * (k1 + 2 * k2 + 2 * k3 + k4)


def integrate_streamline(seed, dt=0.05, n_steps=400, domain=DOMAIN, R=R):
    """Trace one streamline forward and backward from a seed point."""
    pts = [np.array(seed, dtype=float)]

    def march(direction):
        p = np.array(seed, dtype=float)
        for _ in range(n_steps):
            p = rk4_step(p, direction * dt)
            if np.any(np.abs(p) > domain + 0.5):
                break
            if np.linalg.norm(p) < R * 1.001:
                break
            pts.append(p.copy()) if direction > 0 else pts.insert(0, p.copy())

    march(+1)
    march(-1)
    return np.array(pts)


rng = np.random.default_rng(7)

# Seeds on an upstream disk (x = -DOMAIN)
radii = np.linspace(0.35 * R, 2.2 * R, 6)
angles = np.linspace(0, 2 * np.pi, 8, endpoint=False)
seeds = []
for rad in radii:
    for ang in angles:
        seeds.append([-DOMAIN, rad * np.cos(ang), rad * np.sin(ang)])
seeds = np.array(seeds[:N_STREAM])

streamlines = [integrate_streamline(s) for s in seeds]
print(f"Integrated {len(streamlines)} streamlines")

## 5. Interactive 3D scene (sphere + streamlines + midplane speed)

In [ ]:
# Mid-plane (z=0) speed field for a translucent slice
xg = np.linspace(-DOMAIN, DOMAIN, N_GRID)
yg = np.linspace(-DOMAIN, DOMAIN, N_GRID)
Xg, Yg = np.meshgrid(xg, yg, indexing="xy")
Zg = np.zeros_like(Xg)
spd = speed(Xg, Yg, Zg)
spd = np.where(np.sqrt(Xg**2 + Yg**2) < R, np.nan, spd)

fig = go.Figure()

# Sphere colored by Cp
fig.add_trace(
    go.Surface(
        x=xs, y=ys, z=zs,
        surfacecolor=cp_surf,
        colorscale="RdBu_r",
        cmin=-1.25, cmax=1.0,
        colorbar=dict(title="Cp", x=1.02, len=0.6),
        name="Sphere Cp",
        showscale=True,
        opacity=1.0,
        hovertemplate="Cp=%{surfacecolor:.3f}<extra></extra>",
    )
)

# Streamlines colored by local speed
for i, line in enumerate(streamlines):
    s = speed(line[:, 0], line[:, 1], line[:, 2])
    fig.add_trace(
        go.Scatter3d(
            x=line[:, 0], y=line[:, 1], z=line[:, 2],
            mode="lines",
            line=dict(
                width=3,
                color=s,
                colorscale="Viridis",
                cmin=0.4,
                cmax=1.6,
            ),
            showlegend=False,
            hoverinfo="skip",
            name=f"stream {i}",
        )
    )

# Freestream direction arrow
fig.add_trace(
    go.Cone(
        x=[-DOMAIN + 0.4], y=[0], z=[0],
        u=[U_INF], v=[0], w=[0],
        sizemode="absolute",
        sizeref=0.6,
        anchor="tail",
        colorscale=[[0, "#334155"], [1, "#334155"]],
        showscale=False,
        name="U∞",
    )
)

fig.update_layout(
    title="Potential flow past a sphere — 3D streamlines & surface Cp",
    scene=dict(
        xaxis_title="x (flow)",
        yaxis_title="y",
        zaxis_title="z",
        aspectmode="cube",
        xaxis=dict(range=[-DOMAIN, DOMAIN], backgroundcolor="#f8fafc"),
        yaxis=dict(range=[-DOMAIN, DOMAIN], backgroundcolor="#f8fafc"),
        zaxis=dict(range=[-DOMAIN, DOMAIN], backgroundcolor="#f8fafc"),
        camera=dict(eye=dict(x=1.6, y=1.4, z=1.1)),
    ),
    margin=dict(l=0, r=0, t=50, b=0),
    width=900,
    height=700,
)

fig.show()
fig.write_html("sphere_fluid_flow_3d.html", include_plotlyjs="cdn")
print("Saved interactive plot → sphere_fluid_flow_3d.html")

## 6. Mid-plane speed & pressure contour

In [ ]:
p_plane = pressure(Xg, Yg, Zg)
p_plane = np.where(np.sqrt(Xg**2 + Yg**2) < R, np.nan, p_plane)

fig2 = make_subplots(
    rows=1, cols=2,
    subplot_titles=("|V| / U∞ on z=0", "Pressure on z=0"),
    horizontal_spacing=0.12,
)

fig2.add_trace(
    go.Heatmap(x=xg, y=yg, z=spd / U_INF, colorscale="Viridis",
               colorbar=dict(title="|V|/U", x=0.45, len=0.75),
               zmin=0.2, zmax=1.7),
    row=1, col=1,
)
fig2.add_trace(
    go.Heatmap(x=xg, y=yg, z=p_plane, colorscale="RdBu_r",
               colorbar=dict(title="p", x=1.02, len=0.75)),
    row=1, col=2,
)

# Outline the sphere
phi_c = np.linspace(0, 2 * np.pi, 200)
for col in (1, 2):
    fig2.add_trace(
        go.Scatter(
            x=R * np.cos(phi_c), y=R * np.sin(phi_c),
            mode="lines", line=dict(color="black", width=2),
            showlegend=False, hoverinfo="skip",
        ),
        row=1, col=col,
    )

fig2.update_xaxes(title_text="x", scaleanchor="y", scaleratio=1, row=1, col=1)
fig2.update_xaxes(title_text="x", scaleanchor="y", scaleratio=1, row=1, col=2)
fig2.update_yaxes(title_text="y", row=1, col=1)
fig2.update_yaxes(title_text="y", row=1, col=2)
fig2.update_layout(height=480, width=950, title_text="Mid-plane flow quantities")
fig2.show()

## 7. Tracer-particle animation (flow past the sphere)

In [ ]:
def seed_particles(n=N_PARTICLES, domain=DOMAIN, R=R, rng=rng):
    """Release particles on an upstream plane with a soft radial bias."""
    rad = R * (0.2 + 2.4 * np.sqrt(rng.random(n)))
    ang = rng.uniform(0, 2 * np.pi, n)
    x = np.full(n, -domain)
    y = rad * np.cos(ang)
    z = rad * np.sin(ang)
    return np.column_stack([x, y, z])


def advect_particles(positions, n_frames=N_FRAMES, dt=DT, domain=DOMAIN):
    traj = np.zeros((n_frames, len(positions), 3))
    pos = positions.copy()
    for f in range(n_frames):
        traj[f] = pos
        for i in range(len(pos)):
            p = pos[i]
            # Recycle particles that leave the box
            if p[0] > domain or np.linalg.norm(p[1:]) > domain:
                rad = R * (0.2 + 2.4 * np.sqrt(rng.random()))
                ang = rng.uniform(0, 2 * np.pi)
                pos[i] = [-domain, rad * np.cos(ang), rad * np.sin(ang)]
                continue
            pos[i] = rk4_step(p, dt)
            # Soft collision: push out if inside sphere
            nrm = np.linalg.norm(pos[i])
            if nrm < R * 1.01:
                pos[i] = pos[i] / nrm * R * 1.02
    return traj


particles0 = seed_particles()
traj = advect_particles(particles0)
print(f"Particle trajectories: {traj.shape}")

In [ ]:
# Build Plotly animation frames
frames = []
for f in range(N_FRAMES):
    pts = traj[f]
    s = speed(pts[:, 0], pts[:, 1], pts[:, 2])
    frames.append(
        go.Frame(
            data=[
                go.Scatter3d(
                    x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
                    mode="markers",
                    marker=dict(size=3, color=s, colorscale="Viridis",
                                cmin=0.4, cmax=1.6, opacity=0.85),
                )
            ],
            name=str(f),
        )
    )

fig3 = go.Figure(
    data=[
        go.Scatter3d(
            x=traj[0, :, 0], y=traj[0, :, 1], z=traj[0, :, 2],
            mode="markers",
            marker=dict(size=3, color=speed(traj[0, :, 0], traj[0, :, 1], traj[0, :, 2]),
                        colorscale="Viridis", cmin=0.4, cmax=1.6, opacity=0.85),
            name="tracers",
        ),
        go.Surface(
            x=xs, y=ys, z=zs,
            surfacecolor=cp_surf,
            colorscale="RdBu_r",
            cmin=-1.25, cmax=1.0,
            showscale=False,
            opacity=0.95,
            name="sphere",
        ),
    ],
    frames=frames,
)

fig3.update_layout(
    title="Tracer particles advected past the sphere",
    scene=dict(
        xaxis_title="x", yaxis_title="y", zaxis_title="z",
        aspectmode="cube",
        xaxis=dict(range=[-DOMAIN, DOMAIN]),
        yaxis=dict(range=[-DOMAIN, DOMAIN]),
        zaxis=dict(range=[-DOMAIN, DOMAIN]),
        camera=dict(eye=dict(x=1.5, y=1.3, z=1.0)),
    ),
    updatemenus=[{
        "type": "buttons",
        "showactive": False,
        "y": 0,
        "x": 0.1,
        "buttons": [
            {"label": "Play", "method": "animate",
             "args": [None, {"frame": {"duration": 40, "redraw": True},
                             "fromcurrent": True, "mode": "immediate"}]},
            {"label": "Pause", "method": "animate",
             "args": [[None], {"frame": {"duration": 0, "redraw": False},
                               "mode": "immediate"}]},
        ],
    }],
    sliders=[{
        "steps": [
            {"args": [[str(k)], {"frame": {"duration": 0, "redraw": True},
                                "mode": "immediate"}],
             "label": str(k), "method": "animate"}
            for k in range(N_FRAMES)
        ],
        "x": 0.1, "len": 0.85, "y": 0,
    }],
    width=900,
    height=700,
    margin=dict(l=0, r=0, t=50, b=40),
)

fig3.show()
fig3.write_html("sphere_fluid_particles.html", include_plotlyjs="cdn")
print("Saved animation → sphere_fluid_particles.html")

## 8. Quick checks

- Far-field velocity → $U_\infty$
- Equator surface speed → $1.5\,U_\infty$
- Stagnation points at $(\pm R, 0, 0)$ → $|V|=0$, $C_p=1$

In [ ]:
checks = {
    "far-field speed": float(speed(20.0, 0.0, 0.0)),
    "equator speed": float(speed(0.0, R + 1e-8, 0.0)),
    "nose stagnation": float(speed(R + 1e-8, 0.0, 0.0)),
    "tail stagnation": float(speed(-(R + 1e-8), 0.0, 0.0)),
    "Cp nose (analytic)": float(1.0 - (1.5 * 0.0) ** 2),
    "Cp equator (analytic)": float(1.0 - (1.5 * 1.0) ** 2),
}
for k, v in checks.items():
    print(f"{k:28s} {v: .6f}")

assert abs(checks["far-field speed"] - U_INF) < 0.01
assert abs(checks["equator speed"] - 1.5 * U_INF) < 0.02
assert checks["nose stagnation"] < 0.05
print("\nAll analytic checks passed.")